# Full-Stack Project Builder | Hierarchical Multi-Agent Teams

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [3]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [4]:
model = ChatOpenAI(model="gpt-4o")

In [5]:
# --- Research Team ---

# Web search tool (free, no API key; requires `duckduckgo-search` package)
search_tool = DuckDuckGoSearchResults(name="web_search")

# Analysis tool -- reuses the module-level model instance
@tool
def analyze_data(data: str) -> str:
    """Analyze the provided data and return structured insights with key findings."""
    response = model.invoke(
        f"You are a data analyst. Analyze the following data and provide:\n"
        f"1. Key findings (bullet points)\n"
        f"2. Patterns or trends\n"
        f"3. Recommendations\n\n"
        f"Data:\n{data}"
    )
    return response.content

# Note: `name` is required for multi-agent systems (supervisor identifies agents by name).
# If `create_agent` does not accept `name`, use `create_react_agent` from `langgraph.prebuilt` instead.
web_researcher = create_agent(
    model=model, tools=[search_tool], name="web_researcher",
    system_prompt="You search the web for information. Return detailed findings."
)

data_analyst = create_agent(
    model=model, tools=[analyze_data], name="data_analyst",
    system_prompt="You analyze data and provide structured insights."
)

# name must be passed to BOTH create_supervisor and compile() for hierarchical setups
research_team = create_supervisor(
    agents=[web_researcher, data_analyst],
    model=model,
    prompt="You lead the research team. Coordinate web research and data analysis.",
    name="research_team",
).compile(name="research_team")

In [6]:
# --- Engineering Team ---

# Frontend tool -- reuses the module-level model instance
@tool
def write_frontend(spec: str) -> str:
    """Generate frontend code (HTML/CSS/JavaScript) based on a specification."""
    response = model.invoke(
        f"You are a frontend developer. Write clean, production-ready HTML/CSS/JS code for:\n\n{spec}\n\n"
        f"Return only the code with brief inline comments."
    )
    return response.content

# Backend tool -- reuses the module-level model instance
@tool
def write_backend(spec: str) -> str:
    """Generate backend code (Python/FastAPI) based on a specification."""
    response = model.invoke(
        f"You are a backend developer. Write clean, production-ready Python (FastAPI) code for:\n\n{spec}\n\n"
        f"Return only the code with brief inline comments."
    )
    return response.content

frontend_dev = create_agent(
    model=model, tools=[write_frontend], name="frontend_dev",
    system_prompt="You write frontend code (HTML, CSS, JavaScript)."
)

backend_dev = create_agent(
    model=model, tools=[write_backend], name="backend_dev",
    system_prompt="You write backend code (Python, FastAPI, databases)."
)

# name must be passed to BOTH create_supervisor and compile() for hierarchical setups
engineering_team = create_supervisor(
    agents=[frontend_dev, backend_dev],
    model=model,
    prompt="You lead the engineering team. Coordinate frontend and backend development.",
    name="engineering_team",
).compile(name="engineering_team")

In [7]:
# --- Top-Level Supervisor ---
# create_supervisor accepts both compiled and uncompiled graphs as agents
top_supervisor = create_supervisor(
    agents=[research_team, engineering_team],
    model=model,
    prompt=(
        "You are the project manager overseeing research and engineering teams. "
        "Break down the user's request and delegate to the appropriate team. "
        "Use the research team for information gathering and analysis. "
        "Use the engineering team for building and coding."
    )
)

In [8]:
app = top_supervisor.compile()

In [9]:
# Plot the workflow
plot_mermaid(app)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	research_team(research_team)
	engineering_team(engineering_team)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	engineering_team --> supervisor;
	research_team --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> engineering_team;
	supervisor -.-> research_team;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
result = app.invoke({
    "messages": [{
        "role": "user",
        "content": "Research the best practices for REST API design, then build a sample API spec"
    }]
})

In [11]:
print(result["messages"][-1].content)

The research team has provided resources on best practices for REST API design, and the engineering team has crafted a sample API specification for an E-Commerce Product Catalog. If you need further development or adjustments to the specification, feel free to reach out!


In [12]:
stream_invoke(app, {
    "messages": [{
        "role": "user",
        "content": "Research the best practices for REST API design, then build a sample API spec"
    }]
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

┌─ HUMAN
│ Research the best practices for REST API design, then build a sample API spec
└────────────────────────────────────────

┌─ AI (supervisor)
│ → tool: transfer_to_research_team({})
└────────────────────────────────────────

┌─ TOOL (transfer_to_research_team)
│ Successfully transferred to research_team
└────────────────────────────────────────

┌─ AI (supervisor)
│ Here's a summary of the best practices for REST API design along with a suggestion for a sample API specification:
│ 
│ ### Best Practices for REST API Design
│ 
│ 1. **Use HTTP Methods Correctly**: Ensure the use of HTTP methods (GET, POST, PUT, DELETE, etc.) align with their intended purposes. For instance, use GET to retrieve data, POST to create data, PUT to update, and DELETE to remove data.
│ 
│ 2. **Organize Resources and End

{'messages': [HumanMessage(content='Research the best practices for REST API design, then build a sample API spec', additional_kwargs={}, response_metadata={}, id='38b44eda-724d-423a-b7d9-b3b7101c617a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 120, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f4707bdfbe', 'id': 'chatcmpl-DJeL8KzcfRKDPeec2BC0LmYzHa1Z2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019cf159-0420-7f90-b8cb-196aaf6d5996-0', tool_calls=[{'name': 'transfer_to_research_team', 'args': {}, 'id': 'call_4SyV3Z0JkpbKrNY2mZHkgnDH', 'type': 'tool_call'}], inv